# [Qdrant VectorDB](https://qdrant.tech/)


## 1. Qdrant란?

Qdrant는 고성능 벡터 유사도 검색을 위한 오픈소스 벡터 데이터베이스입니다.


### 주요 특징
- **고성능**: Rust로 작성되어 매우 빠른 검색 속도 제공
- **확장 가능한 필터링**: 복잡한 조건을 사용한 필터링 지원
- **페이로드 저장**: 벡터와 함께 JSON 형태의 메타데이터 저장
- **REST API & gRPC**: 다양한 프로토콜을 통한 접근 가능
- **클라우드 네이티브**: Docker, Kubernetes 등 컨테이너 환경에 최적화
- **LangChain 통합**: LangChain과 완벽하게 통합되어 RAG 시스템 구축 가능


### 사용 사례
- 시맨틱 검색 (Semantic Search)
- 추천 시스템 (Recommendation Systems)
- 이미지/비디오 검색
- RAG (Retrieval Augmented Generation) 시스템
- 중복 탐지 (Duplicate Detection)
- 이상 탐지 (Anomaly Detection)


## 2. Qdrant vs 다른 벡터 데이터베이스


### Qdrant의 장점
- **고성능**: Rust 기반으로 매우 빠른 검색 속도
- **풍부한 필터링**: SQL과 유사한 강력한 필터링 기능
- **확장성**: 수평적 확장 지원 (Horizontal Scaling)
- **관리 편의성**: 직관적인 REST API와 Web UI 제공
- **실시간 업데이트**: 실시간으로 데이터 추가/수정/삭제 가능
- **클라우드 지원**: Qdrant Cloud를 통한 관리형 서비스 제공


### 다른 솔루션과의 비교
- **Chroma**: 간단한 설정, 메모리 기반, 소규모 프로젝트에 적합
- **pgvector**: 기존 DB 활용, 중간 규모, 엔터프라이즈 환경
- **Qdrant**: 고성능, 풍부한 필터링, 중소규모에서 대규모까지 적합
- **Milvus**: 대규모 분산 환경, 높은 성능, 복잡한 설정


## 3. 설치 및 설정

Qdrant를 사용하기 위한 설치와 설정 방법을 알아보겠습니다.


### Qdrant 클라이언트 연결


In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333)

# Qdrant Cloud 연결
# client = QdrantClient(
#     url="https://your-cluster.qdrant.io",
#     api_key="your-api-key"
# )

print("Qdrant 클라이언트 연결 완료")


## [4. LangChain과 Qdrant 통합](https://docs.langchain.com/oss/python/langchain/knowledge-base#qdrant)

실제 텍스트 파일을 로드하여 Qdrant에 저장하고 검색하는 완전한 예제를 만들어보겠습니다.


### 환경 변수 로드


In [ ]:
from dotenv import load_dotenv

load_dotenv()


### 텍스트 파일 로드


In [ ]:
from langchain.document_loaders import TextLoader

loader = TextLoader("./data/rag-keywords.txt", encoding="utf-8")
documents = loader.load()
print(f"파일 로드 완료: {len(documents)}개 문서")


### 텍스트 분할


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

splits = text_splitter.split_documents(documents)
print(f"텍스트 분할 완료: {len(splits)}개 청크")


### 임베딩 모델 설정


In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


### Qdrant 벡터스토어 생성


In [ ]:
from langchain_qdrant import QdrantVectorStore

# Qdrant 벡터스토어 생성
vectorstore = QdrantVectorStore.from_documents(
    documents=splits,           # 분할된 문서
    embedding=embeddings,       # 임베딩 함수
    location=":memory:",        # 인메모리 모드 (또는 "http://localhost:6333")
    collection_name="rag_keywords",  # 컬렉션 이름
)

print("Qdrant 벡터스토어 생성 완료")


## 5. 데이터 저장 및 검색

Qdrant를 사용하여 문서를 저장하고 검색하는 방법을 알아보겠습니다.


### 샘플 문서 데이터


In [ ]:
from langchain.schema import Document

sample_documents = [
    Document(
        page_content="인공지능은 인간의 지능을 모방하는 기술입니다.",
        metadata={"category": "AI", "topic": "기본개념", "source": "ai_basics.txt"}
    ),
    Document(
        page_content="머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야입니다.",
        metadata={"category": "ML", "topic": "학습방법", "source": "ml_intro.txt"}
    ),
    Document(
        page_content="딥러닝은 신경망을 사용하는 머신러닝의 하위 분야입니다.",
        metadata={"category": "DL", "topic": "신경망", "source": "dl_guide.txt"}
    ),
    Document(
        page_content="자연어처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
        metadata={"category": "NLP", "topic": "언어처리", "source": "nlp_handbook.txt"}
    )
]

print(f"샘플 문서 {len(sample_documents)}개가 준비되었습니다.")
for i, doc in enumerate(sample_documents, 1):
    print(f"{i}. {doc.page_content[:30]}... (카테고리: {doc.metadata['category']})")


### 추가 벡터스토어 생성 (샘플 문서용)


In [ ]:
# 샘플 문서를 위한 별도의 벡터스토어
sample_vectorstore = QdrantVectorStore.from_documents(
    documents=sample_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="sample_docs",
)

print("샘플 벡터스토어 생성 완료")


### 유사도 검색


In [ ]:
def similarity_search_example(vectorstore, query, k=3):
    """유사도 검색 예제"""
    
    try:
        # 기본 유사도 검색
        results = vectorstore.similarity_search(query, k=k)
        
        print(f"검색 쿼리: '{query}'")
        print(f"검색 결과 ({len(results)}개):")
        print("-" * 50)
        
        for i, doc in enumerate(results, 1):
            print(f"{i}. {doc.page_content}")
            print(f"   메타데이터: {doc.metadata}")
            print()
            
        return results
    except Exception as e:
        print(f"검색 중 오류 발생: {e}")
        return []


In [ ]:
similarity_search_example(sample_vectorstore, "신경망에 대해 알려주세요")


### 유사도 점수와 함께 검색


In [ ]:
def similarity_search_with_score_example(vectorstore, query, k=3):
    """유사도 점수와 함께 검색"""
    try:
        # 유사도 점수와 함께 검색
        results = vectorstore.similarity_search_with_score(query, k=k)
        
        print(f"검색 쿼리: '{query}' (점수 포함)")
        print(f"검색 결과 ({len(results)}개):")
        print("-" * 50)
        
        for i, (doc, score) in enumerate(results, 1):
            print(f"{i}. {doc.page_content}")
            print(f"   유사도 점수: {score:.4f}")
            print(f"   메타데이터: {doc.metadata}")
            print()
            
        return results
    except Exception as e:
        print(f"검색 중 오류 발생: {e}")
        return []


In [ ]:
similarity_search_with_score_example(sample_vectorstore, "인공지능 기술")


## 6. 고급 검색 기능

Qdrant의 고급 검색 기능들을 알아보겠습니다.


### 메타데이터 필터링 검색

Qdrant는 강력한 필터링 기능을 제공합니다.


In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def filtered_search_example(vectorstore, query, filter_dict, k=3):
    """메타데이터로 필터링된 검색"""
    try:
        # LangChain의 필터 형식 사용
        results = vectorstore.similarity_search(
            query, 
            k=k,
            filter=filter_dict
        )
        
        print(f"검색 쿼리: '{query}'")
        print(f"필터 조건: {filter_dict}")
        print(f"검색 결과 ({len(results)}개):")
        print("-" * 50)
        
        for i, doc in enumerate(results, 1):
            print(f"{i}. {doc.page_content}")
            print(f"   메타데이터: {doc.metadata}")
            print()
            
        return results
    except Exception as e:
        print(f"필터링 검색 중 오류 발생: {e}")
        return []


In [ ]:
# AI 카테고리만 검색
filtered_search_example(sample_vectorstore, "인공지능", {"category": "AI"})


### MMR (Maximal Marginal Relevance) 검색

MMR은 관련성과 다양성을 모두 고려한 검색 방법입니다.


In [ ]:
def mmr_search_example(vectorstore, query, k=3, fetch_k=10, lambda_mult=0.5):
    """MMR 검색 예제"""
    try:
        # MMR 검색
        # lambda_mult: 0에 가까우면 다양성 우선, 1에 가까우면 관련성 우선
        results = vectorstore.max_marginal_relevance_search(
            query, 
            k=k,
            fetch_k=fetch_k,  # 먼저 가져올 후보 문서 수
            lambda_mult=lambda_mult  # 관련성 vs 다양성 비율
        )
        
        print(f"검색 쿼리: '{query}'")
        print(f"MMR 검색 결과 (lambda={lambda_mult}):")
        print("-" * 50)
        
        for i, doc in enumerate(results, 1):
            print(f"{i}. {doc.page_content}")
            print(f"   메타데이터: {doc.metadata}")
            print()
            
        return results
    except Exception as e:
        print(f"MMR 검색 중 오류 발생: {e}")
        return []


In [ ]:
mmr_search_example(sample_vectorstore, "머신러닝 학습", k=3, lambda_mult=0.5)


## 7. RetrievalQA를 이용한 질의응답

Qdrant와 LLM을 결합하여 실제 질의응답 시스템을 구축해보겠습니다.


### LLM 설정


In [ ]:
from langchain_openai import ChatOpenAI

# LLM 설정
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,  # 일관된 답변을 위해 0으로 설정
)


### RAG 체인 구성


In [ ]:
from langchain.chains import RetrievalQA

# RAG 체인 구성
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # 모든 문서를 하나의 프롬프트에 포함
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2})
)

print("RAG 체인 구성 완료")


### 질문 답변 실행


In [ ]:
# 질문 답변 실행
question = "RAG에 대해 설명해주세요"
answer = qa_chain.invoke(question)

print(f"질문: {answer['query']}")
print(f"답변: {answer['result']}")


### 다양한 질문 테스트


In [ ]:
def ask_question(qa_chain, question):
    """질문에 대한 답변을 출력하는 함수"""
    print(f"\n{'='*60}")
    print(f"질문: {question}")
    print(f"{'='*60}")
    
    answer = qa_chain.invoke(question)
    print(f"답변: {answer['result']}")
    print(f"{'='*60}\n")
    
    return answer


In [ ]:
# 여러 질문 테스트
questions = [
    "벡터 데이터베이스란 무엇인가요?",
    "임베딩의 역할은 무엇인가요?",
    "검색 증강 생성이 필요한 이유는?",
]

for q in questions:
    ask_question(qa_chain, q)


## 8. Qdrant 클라이언트 직접 사용하기

LangChain 없이 Qdrant 클라이언트를 직접 사용하는 방법도 알아보겠습니다.


### 컬렉션 생성


In [ ]:
from qdrant_client.models import Distance, VectorParams

# 새로운 컬렉션 생성
collection_name = "my_collection"

try:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=1536,  # text-embedding-3-small의 차원
            distance=Distance.COSINE  # 코사인 거리 사용
        )
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")
except Exception as e:
    print(f"컬렉션 생성 중 오류: {e}")


### 데이터 추가


In [ ]:
from qdrant_client.models import PointStruct
import uuid

# 텍스트와 메타데이터
texts = [
    "Qdrant는 고성능 벡터 검색 엔진입니다.",
    "Python으로 벡터 데이터베이스를 쉽게 사용할 수 있습니다.",
]

# 임베딩 생성
vectors = embeddings.embed_documents(texts)

# 포인트 생성 및 추가
points = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=vector,
        payload={"text": text, "index": i}
    )
    for i, (text, vector) in enumerate(zip(texts, vectors))
]

try:
    client.upsert(
        collection_name=collection_name,
        points=points
    )
    print(f"{len(points)}개의 포인트 추가 완료")
except Exception as e:
    print(f"데이터 추가 중 오류: {e}")


### 검색 수행


In [ ]:
# 검색 쿼리
query = "벡터 검색"
query_vector = embeddings.embed_query(query)

# 검색 실행
try:
    search_result = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=2
    )
    
    print(f"검색 쿼리: '{query}'")
    print("검색 결과:")
    print("-" * 50)
    
    for i, hit in enumerate(search_result, 1):
        print(f"{i}. {hit.payload['text']}")
        print(f"   점수: {hit.score:.4f}")
        print(f"   메타데이터: {hit.payload}")
        print()
        
except Exception as e:
    print(f"검색 중 오류: {e}")


## 9. 성능 최적화 팁

Qdrant를 효율적으로 사용하기 위한 팁들입니다.


### 1. 인덱싱 최적화

```python
# HNSW 인덱스 파라미터 설정
from qdrant_client.models import HnswConfigDiff

client.update_collection(
    collection_name="my_collection",
    hnsw_config=HnswConfigDiff(
        m=16,  # 그래프의 연결 수 (기본값: 16)
        ef_construct=100,  # 인덱스 구축 시 탐색 깊이 (기본값: 100)
    )
)
```

### 2. 검색 성능 튜닝

```python
# 검색 시 정확도와 속도 조절
search_result = client.search(
    collection_name="my_collection",
    query_vector=query_vector,
    limit=10,
    search_params={
        "hnsw_ef": 128,  # 검색 시 탐색 깊이 (높을수록 정확하지만 느림)
    }
)
```

### 3. 배치 처리

```python
# 대량의 데이터를 배치로 추가
client.upsert(
    collection_name="my_collection",
    points=points,  # 여러 포인트를 한 번에
    wait=False  # 비동기 처리
)
```

### 4. 페이로드 인덱싱

```python
# 자주 필터링하는 필드에 인덱스 생성
from qdrant_client.models import PayloadSchemaType

client.create_payload_index(
    collection_name="my_collection",
    field_name="category",
    field_schema=PayloadSchemaType.KEYWORD
)
```


## 10. 정리

### Qdrant의 주요 장점
1. **고성능**: Rust로 작성되어 매우 빠른 검색 속도
2. **풍부한 기능**: 강력한 필터링, 페이로드 저장 등
3. **확장성**: 수평적 확장 지원
4. **사용 편의성**: 직관적인 API와 Web UI
5. **LangChain 통합**: RAG 시스템 구축에 최적화

### 언제 Qdrant를 사용할까?
- 고성능 벡터 검색이 필요할 때
- 복잡한 필터링 조건이 필요할 때
- 클라우드 네이티브 환경에서 운영할 때
- 중소규모에서 대규모 시스템까지

### 다음 단계
- Qdrant Cloud 사용해보기
- 분산 환경 구성
- 프로덕션 환경 배포
- 모니터링 및 최적화
